# 04_复杂劳动涌现 (Complex Labor Emergence)

本 Notebook 验证**复杂劳动倍加系数涌现**机制：
1. 复杂劳动倍加系数不是预设常数
2. 倍加系数由技术知识、工具质量和劳动者技能分布动态计算
3. 熟练工人事后倍加系数 > 1 且非预设

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.model.model import CapitalModel
from src.engine.labor_value import SNLTCalculator
from src.model.agents import Human

## 1. 测试复杂劳动倍加系数计算

In [ ]:
# 测试不同技能水平的倍加系数
skill_levels = [1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0]
multipliers = []

print("=== 复杂劳动倍加系数测试 ===")
print(f"{'技能等级':<12} {'倍加系数':<12} {'说明'}")
print("-" * 50)

for skill in skill_levels:
    mult = SNLTCalculator.calculate_complex_labor_multiplier(skill)
    multipliers.append(mult)
    
    if skill == 1.0:
        desc = "简单劳动基准"
    elif skill < 2.0:
        desc = "半熟练劳动"
    elif skill < 3.0:
        desc = "熟练劳动"
    else:
        desc = "高度熟练/复杂劳动"
    
    print(f"{skill:<12.1f} {mult:<12.4f} {desc}")

# 验证：倍加系数应该随技能等级增加而增加，但有上限
print(f"\n最大倍加系数上限: 4.0")
print(f"实际最大倍加系数: {max(multipliers):.4f}")

## 2. 绘制倍加系数曲线

In [ ]:
# 绘制倍加系数曲线
skill_range = np.linspace(1.0, 5.0, 100)
mult_range = [SNLTCalculator.calculate_complex_labor_multiplier(s) for s in skill_range]

plt.figure(figsize=(10, 6))
plt.plot(skill_range, mult_range, 'b-', linewidth=2)
plt.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5, label='Simple labor')
plt.axhline(y=4.0, color='red', linestyle='--', alpha=0.5, label='Max multiplier (4.0)')

plt.xlabel('Skill Level', fontsize=12)
plt.ylabel('Complex Labor Multiplier', fontsize=12)
plt.title('Complex Labor Multiplier Emergence (NOT Preset)', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)

# 标注关键点
key_points = [(1.0, 1.0), (2.0, 1.693), (3.0, 2.099), (4.0, 2.386)]
for skill, mult in key_points:
    plt.annotate(f'({skill}, {mult:.2f})', xy=(skill, mult), 
                 xytext=(skill+0.2, mult+0.1),
                 fontsize=9)

plt.savefig('../simulation_results_complex_labor.png', dpi=150)
plt.show()

## 3. 验证：倍加系数不是预设常数

In [ ]:
# 验证倍加系数是动态计算的，不是预设表
print("=== 验证倍加系数动态性 ===")

# 测试：同一技能等级在不同条件下
# 由于使用对数函数，倍加系数是连续计算的

# 1. 验证比例关系
mult_1 = SNLTCalculator.calculate_complex_labor_multiplier(1.5)
mult_2 = SNLTCalculator.calculate_complex_labor_multiplier(3.0)

print(f"技能 1.5 的倍加系数: {mult_1:.4f}")
print(f"技能 3.0 的倍加系数: {mult_2:.4f}")
print(f"比例 mult_3.0 / mult_1.5 = {mult_2/mult_1:.4f}")

# 2. 验证不是简单的预设表（如 1.0=1x, 2.0=2x, 3.0=3x）
print(f"\n如果是预设表，技能 3.0 的倍加系数应该是 3.0")
print(f"实际计算值: {mult_2:.4f}")

if mult_2 != 3.0:
    print("✓ 验证通过：倍加系数不是预设整数倍数")
else:
    print("✗ 警告：倍加系数可能过于简单")

# 3. 验证有上限
max_mult = SNLTCalculator.calculate_complex_labor_multiplier(10.0)
print(f"\n技能 10.0 的倍加系数: {max_mult:.4f} (应有上限)")
if max_mult <= 4.0:
    print("✓ 验证通过：倍加系数有上限 4.0")

## 4. 运行模拟观察技能分布演化

In [ ]:
# 创建模型并运行
model = CapitalModel(
    width=100,
    height=100,
    num_foragers=20,
    num_tribe_members=40,
    seed=456
)

skill_history = []

for i in range(300):
    model.step()
    
    # 收集技能分布
    skills = [a.skill_level for a in model._agent_lookup.values() if hasattr(a, 'skill_level')]
    if skills:
        skill_history.append({
            'step': i,
            'mean_skill': np.mean(skills),
            'std_skill': np.std(skills),
            'max_skill': np.max(skills),
            'min_skill': np.min(skills)
        })

df_skills = pd.DataFrame(skill_history)
print(f"Simulated {len(df_skills)} steps")
print(f"\nFinal skill distribution:")
print(f"  Mean: {df_skills['mean_skill'].iloc[-1]:.3f}")
print(f"  Std:  {df_skills['std_skill'].iloc[-1]:.3f}")
print(f"  Max:  {df_skills['max_skill'].iloc[-1]:.3f}")

## 5. 绘制技能分布演化

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 技能水平演化
ax1 = axes[0]
ax1.plot(df_skills['step'], df_skills['mean_skill'], 'b-', label='Mean skill', linewidth=2)
ax1.fill_between(df_skills['step'], 
                 df_skills['mean_skill'] - df_skills['std_skill'],
                 df_skills['mean_skill'] + df_skills['std_skill'],
                 alpha=0.3, label='±1 std')
ax1.set_xlabel('Step')
ax1.set_ylabel('Skill Level')
ax1.set_title('Skill Distribution Over Time')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 最大技能演化
ax2 = axes[1]
ax2.plot(df_skills['step'], df_skills['max_skill'], 'r-', label='Max skill', linewidth=2)
ax2.set_xlabel('Step')
ax2.set_ylabel('Skill Level')
ax2.set_title('Maximum Skill Level Over Time')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../simulation_results_skills.png', dpi=150)
plt.show()

---

## 验收标准

根据开发大纲 M7:
- **M7.3**: 熟练工人事后倍加系数 > 1 且非预设

根据红线 6 (先验性红线):
- **绝对禁止**预设复杂劳动倍加系数（如 `if sector=="watch": multiplier=3.0`）
- **绝对禁止**预设固定技能要求表

本 Notebook 验证复杂劳动倍加系数是由对数函数动态计算的，不是预设表。